In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI  # to talk with LLM
from langchain_core.documents import Document # to prepare text document
from langchain_text_splitters import RecursiveCharacterTextSplitter # to convert document to chunks
from langchain_openai import OpenAIEmbeddings # To convert chunks to vectors
from langchain_community.vectorstores import Chroma # to store vectors to Chromadb


In [ ]:
llm=ChatOpenAI(model="gpt-5-nano",temperature=0)


### step 1 - Extracting text from pdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path="./pyspark.pdf"

loader=PyPDFLoader(pdf_path)

docs=loader.load()
docs

#### STEP 2 : Create Chunks

In [ ]:
##Split the document to chunks

splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)

chunks=splitter.split_documents(docs)

In [ ]:
len(chunks)

In [ ]:
### Since we got 781 chunks which means 781 vectores will be created

### STEP 3 : Create Embedding Model

In [ ]:
embedding_model=OpenAIEmbeddings(model="text-embedding-3-small")

### step 4 : Create Vector Store

In [ ]:
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./Vector/"  ### this can be used if we can self the vector db instead of building in-memory vector db
)

### Step 5 : Semantic Search

In [ ]:
vectorstore.similarity_search("What are internals of spark. give me a overview",k=20)

In [ ]:
context=vectorstore.similarity_search("What are internals of spark. give me a overview",k=20)

In [ ]:
#### Talk to LLM
response = llm.invoke(f"what are internals of spark. you can use the following context {context}")
print(response.content)

In [ ]:
#### since we have pyisical vector db we dont have to create in-memory vector db . 
# we can directly connect to created vector db in Vector folder


#### Reuse the vector database

In [ ]:
vectorstore_persist = Chroma( persist_directory="./Vector",
                                embedding_function=embedding_model)

In [ ]:
vectorstore_persist.similarity_search("What are internals of spark. give me a overview",k=20)

In [ ]:
context=vectorstore_persist.similarity_search("What are internals of spark. give me a overview",k=20)

In [ ]:
#### Talk to LLM
response = llm.invoke(f"what are internals of spark. you can use the following context {context}")
print(response.content)

In [ ]:
print("RAG Implementation Done")